# 01 — Dataset Exploration

Inspect and verify the structure files downloaded by `00_data_acquisition.ipynb`.

> **Prerequisites:** run `00_data_acquisition.ipynb` first to populate `data/`.

This notebook does **not** download anything. It only reads what is already on disk.

In [1]:
import sys, os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, ".")

import pandas as pd
from src.parser import load_structure, get_ca_coords_and_seq

In [2]:
# Load the protein manifest
proteins = pd.read_csv("proteins.csv")
proteins

,protein_name,uniprot_id,pdb_xray,pdb_nmr,pdb_cryoem,chain_id,chain_id_af2,nmr_model,notes
0,Barnase,P00648,6PQK,1BNR,NaN,A,NaN,0.0,Added automatically via RCSB search. X-ray res...
1,Protein G (GB1 domain parent),P06654,1IGD,1GB1,9G4T,A,NaN,0.0,Added automatically via RCSB search. X-ray res...
2,Adenylate kinase (E. coli),P69441,7APU,NaN,NaN,A,NaN,NaN,Added automatically via RCSB search. X-ray res...
3,T4 lysozyme,P00720,2O7A,2LC9,8DCS,A,NaN,0.0,Added automatically via RCSB search. X-ray res...
4,Myoglobin,P02185,5YCE,1MYF,NaN,A,NaN,0.0,Added automatically via RCSB search. X-ray res...
...,...,...,...,...,...,...,...,...,...
68,Nucleoporin Nup133,Q8WUM0,1XKS,NaN,NaN,A,NaN,NaN,Added automatically via RCSB search. X-ray res...
69,Importin beta-1,Q14974,1IBR,NaN,9N85,B,NaN,NaN,Added automatically via RCSB search. X-ray res...
70,V-ATPase subunit A,P38606,NaN,NaN,6WLZ,A,NaN,NaN,Added automatically via RCSB search. X-ray res...
71,Photosystem II protein D1,P0C0S1,5AJI,NaN,9E68,A,NaN,NaN,Added automatically via RCSB search. X-ray res...


## Verify all expected files are present

The table shows which CIF files are on disk. Any missing entry must be resolved by re-running notebook 00 before proceeding — the pipeline will silently skip missing files and record them in `results/missing_inputs.csv`.

In [3]:
rows = []

for _, row in proteins.iterrows():
    uid = row["uniprot_id"]
    checks = [
        (
            "X-ray",
            f"data/experimental/{str(row.get('pdb_xray',   '') or '').upper()}.cif",
        ),
        (
            "NMR",
            f"data/experimental/{str(row.get('pdb_nmr',    '') or '').upper()}.cif",
        ),
        (
            "Cryo-EM",
            f"data/experimental/{str(row.get('pdb_cryoem', '') or '').upper()}.cif",
        ),
        ("AF2", f"data/alphafold2/AF2_{uid}.cif"),
        ("OF3", f"data/openfold3/OF3_{uid}.cif"),
    ]
    for method, path in checks:
        fname = os.path.basename(path)
        if fname.split(".")[0].upper() in ("", "NAN", "NONE"):
            continue  # method not available for this protein
        rows.append(
            {
                "protein": row["protein_name"],
                "method": method,
                "file": fname,
                "status": "OK" if os.path.exists(path) else "MISSING",
            }
        )

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()
n_missing = (summary["status"] == "MISSING").sum()
if n_missing == 0:
    print("All expected files present.")
else:
    print(f"{n_missing} file(s) missing — run 00_data_acquisition.ipynb first.")

                                       protein  method           file  status
                                       Barnase   X-ray       6PQK.cif      OK
                                       Barnase     NMR       1BNR.cif      OK
                                       Barnase     AF2 AF2_P00648.cif      OK
                                       Barnase     OF3 OF3_P00648.cif      OK
                 Protein G (GB1 domain parent)   X-ray       1IGD.cif      OK
                 Protein G (GB1 domain parent)     NMR       1GB1.cif      OK
                 Protein G (GB1 domain parent) Cryo-EM       9G4T.cif      OK
                 Protein G (GB1 domain parent)     AF2 AF2_P06654.cif      OK
                 Protein G (GB1 domain parent)     OF3 OF3_P06654.cif      OK
                    Adenylate kinase (E. coli)   X-ray       7APU.cif      OK
                    Adenylate kinase (E. coli)     AF2 AF2_P69441.cif      OK
                    Adenylate kinase (E. coli)     OF3 OF3_P6944

## Sanity check — inspect one structure per type

For each protein, load the X-ray, AF2, and OF3 structures and print basic info:  
number of models, chain list, Cα count, and first 20 residues of the sequence.

In [4]:
for _, row in proteins.iterrows():
    uid = row["uniprot_id"]
    name = row["protein_name"]
    chain_id = str(row["chain_id"])

    to_check = []
    if pd.notna(row.get("pdb_xray")) and str(row["pdb_xray"]).strip():
        to_check.append(
            ("X-ray", f"data/experimental/{str(row['pdb_xray']).upper()}.cif", 0)
        )
    if pd.notna(row.get("pdb_nmr")) and str(row["pdb_nmr"]).strip():
        nmr_model = int(row["nmr_model"]) if pd.notna(row.get("nmr_model")) else 0
        to_check.append(
            ("NMR", f"data/experimental/{str(row['pdb_nmr']).upper()}.cif", nmr_model)
        )
    if pd.notna(row.get("pdb_cryoem")) and str(row["pdb_cryoem"]).strip():
        to_check.append(
            ("Cryo-EM", f"data/experimental/{str(row['pdb_cryoem']).upper()}.cif", 0)
        )
    to_check.append(("AF2", f"data/alphafold2/AF2_{uid}.cif", 0))
    to_check.append(("OF3", f"data/openfold3/OF3_{uid}.cif", 0))

    print(f"\n{'='*55}")
    print(f"  {name}  ({uid})")
    print(f"{'='*55}")

    for label, path, model_idx in to_check:
        if not os.path.exists(path):
            print(f"  {label:<8}  [MISSING] {path}")
            continue
        try:
            struct = load_structure(path)
            coords, seq = get_ca_coords_and_seq(
                struct, model_idx=model_idx, chain_id=chain_id
            )
            print(
                f"  {label:<8}  models={len(struct)}  "
                f"chains={[c.name for c in struct[0]]}  "
                f"Ca={len(coords)}  "
                f"seq={seq[:20]}..."
            )
        except Exception as exc:
            print(f"  {label:<8}  [ERROR] {exc}")


  Barnase  (P00648)
  X-ray     models=1  chains=['A', 'C', 'B', 'D']  Ca=108  seq=VINTFDGVADYLQTYHKLPD...
  NMR       models=20  chains=['A']  Ca=110  seq=AQVINTFDGVADYLQTYHKL...
  AF2       models=1  chains=['A']  Ca=157  seq=MMKMEGIALKKRLSWISVCL...
  OF3       models=1  chains=['A']  Ca=157  seq=MMKMEGIALKKRLSWISVCL...

  Protein G (GB1 domain parent)  (P06654)
  X-ray     models=1  chains=['A']  Ca=61  seq=MTPAVTTYKLVINGKTLKGE...
  NMR       models=60  chains=['A']  Ca=56  seq=MTYKLILNGKTLKGETTTEA...
  Cryo-EM   models=1  chains=['A']  Ca=457  seq=TCRHPLTDLEANEQLGRCED...
  AF2       models=1  chains=['A']  Ca=448  seq=MEKEKKVKYFLRKSAFGLAS...
  OF3       models=1  chains=['A']  Ca=448  seq=MEKEKKVKYFLRKSAFGLAS...

  Adenylate kinase (E. coli)  (P69441)
  X-ray     models=1  chains=['A', 'B']  Ca=214  seq=MRIILLGAPGAGKGTQAQFI...
  AF2       models=1  chains=['A']  Ca=214  seq=MRIILLGAPGAGKGTQAQFI...
  OF3       models=1  chains=['A']  Ca=214  seq=MRIILLGAPGAGKGTQAQFI...

  T4 lysozy

## Sequence length comparison

Check that AF2 and OF3 sequence lengths match the experimental structures.  
Large discrepancies (>10%) may indicate that the prediction covers the full precursor  
sequence (signal peptide, propeptide) while the experimental structure shows only the  
mature chain — this is handled by the coverage filter in the validation step.

In [5]:
length_rows = []

for _, row in proteins.iterrows():
    uid = row["uniprot_id"]
    name = row["protein_name"]
    chain_id = str(row["chain_id"])

    sources = []
    for col, folder, mid in [
        ("pdb_xray", "experimental", 0),
        (
            "pdb_nmr",
            "experimental",
            int(row["nmr_model"]) if pd.notna(row.get("nmr_model")) else 0,
        ),
        ("pdb_cryoem", "experimental", 0),
    ]:
        pdb_id = row.get(col)
        if pd.notna(pdb_id) and str(pdb_id).strip():
            sources.append(
                (
                    col.replace("pdb_", "").upper(),
                    f"data/{folder}/{str(pdb_id).upper()}.cif",
                    mid,
                )
            )
    sources.append(("AF2", f"data/alphafold2/AF2_{uid}.cif", 0))
    sources.append(("OF3", f"data/openfold3/OF3_{uid}.cif", 0))

    for label, path, model_idx in sources:
        if not os.path.exists(path):
            length_rows.append(
                {"protein": name, "source": label, "n_ca": None, "status": "missing"}
            )
            continue
        try:
            struct = load_structure(path)
            coords, _ = get_ca_coords_and_seq(
                struct, model_idx=model_idx, chain_id=chain_id
            )
            length_rows.append(
                {"protein": name, "source": label, "n_ca": len(coords), "status": "ok"}
            )
        except Exception as exc:
            length_rows.append(
                {"protein": name, "source": label, "n_ca": None, "status": str(exc)}
            )

pd.DataFrame(length_rows).pivot(index="protein", columns="source", values="n_ca")

source,AF2,CRYOEM,NMR,OF3,XRAY
protein,,,,,
ATP synthase subunit beta,NaN,469.0,NaN,NaN,467.0
Actin (beta),NaN,NaN,NaN,NaN,15.0
Adenosine A2A receptor,412.0,278.0,NaN,412.0,382.0
Adenylate kinase (E. coli),214.0,NaN,NaN,214.0,214.0
Alpha-synuclein,NaN,61.0,NaN,NaN,11.0
...,...,...,...,...,...
Tumor protein p53,393.0,1476.0,29.0,393.0,14.0
V-ATPase subunit A,617.0,600.0,NaN,617.0,NaN
Vimentin,466.0,NaN,NaN,466.0,39.0
